In [ ]:
params <- list(
    ps_raw = "data/temp/ps_raw.rds"
)

In [ ]:
set.seed(8092003)

In [ ]:
library(phyloseq)
library(dplyr)
library(microbiome)

In [ ]:
ps_raw <- readRDS(params$ps_raw)
ps_core <- microbiome::core(
    ps_raw,
    detection = 1,
    prevalence = 0.05
)
meta <- data.frame(sample_data(ps_raw))

In [ ]:
source("workflow/scripts/asv_vst_normalization.R")

In [ ]:
vst_res <- vst_normalization(ps_core)
ps_core_vst <- vst_res$ps_vst
euc_dist_core <- vst_res$euclidean_dist

In [ ]:
microbiome::summarize_phyloseq(ps_raw)
ntaxa(ps_raw)
ps_raw

In [ ]:
microbiome::summarize_phyloseq(ps_core)
ntaxa(ps_core)
ps_core

In [ ]:
table(meta$batch, meta$lpb_tertile)

In [ ]:
predictors <- c(
    "lpb_tertile",
    "batch",
    # 'd_age_hc',
    "d_sex"
)
predictors <- predictors[meta[predictors] |> sapply(function(x) length(unique(x))) > 1]
rhs <- paste(predictors, collapse = " + ")
rhs

# Taxa overview

In [ ]:
source("workflow/scripts/plots/dendrogram.R")

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 8
)
plot_dendrogram(
    euc_dist_core, ps_core,
    "LBP tertile" = lpb_tertile, Sex = d_sex, Batch = batch,
    taxa_rank = "genus", n_taxa = 30
)

# Beta diversity

## Ordination

Primary coordinates analysis (PCoA) on euclidean coordinate space. Ellipses - 1-SD interval (for bivar normal distr)

In [ ]:
source("workflow/scripts/plots/pcoa.R")

## ASV-level

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 8)
ord_res <- plot_pcoa(ps_core_vst, euc_dist_core, 
    lpb_tertile,
    legend_title="LPB tertile")
pcoa <- ord_res$pcoa

Interesting that PCoA results of High and Low looks similar. Maybe test  normal LBP vs marginal LBP tertile 

In [ ]:
scores <- as.data.frame(pcoa$points)
d2 <- mahalanobis(
    scores,
    colMeans(scores),
    cov(scores)
)
outliers <- which(
    d2 > qchisq(0.975, df = 2)
)

In [ ]:
if (length(names(outliers)) == 0) {
    "There is no outlier samples"
} else {
    names(outliers)
}

### Stat

Test geterogenety of beta-diversity distributions with `vegan::betadisper`

In [ ]:
library(vegan)


In [ ]:
lbp_beta <- betadisper(euc_dist_core, sample_data(ps_core_vst)$lpb_tertile)
lbp_beta$group.distances |> t()

Average distanses from sample to centers of the groups kindly the same (that likely proofs by same size of the ordination ellipses)

significance tested by `vegan::anova`

In [ ]:
anova(lbp_beta, by='margin')

So LBP tertile sets samples have heterogeneous dispersions, which means I cannot truly trust permanova test on whole set. But amount of samples are the same for depression and control sets. So premanova still valid choice

Use permutation analysis for difference (permanova) with `vegan::adonis2` 

In [ ]:
set.seed(08092003)
form_string <- paste("euc_dist_core", "~", rhs)
f <- as.formula(form_string)
(dc_pmnv <- adonis2(f, meta, by = "margin", permutations = how(blocks = meta$batch, nperm = 9999)))

In [ ]:
meta$is_lbp_normal <- meta$lpb_tertile == "Средний ЛПБ"


## Genus level

Firstly aggregate up to genus level

In [ ]:
ps_core_genus <- tax_glom(ps_core, taxrank = "genus")
ps_core_genus

In [ ]:
vst_core_genus <- vst_normalization(ps_core_genus)
ps_core_genus_vst <- vst_core_genus$ps_vst
euc_dist_core_genus <- vst_core_genus$euclidean_dist

In [ ]:
ntaxa(ps_core_genus)

In [ ]:
ord_res <- plot_pcoa(ps_core_genus_vst, euc_dist_core_genus, lpb_tertile)

### Stats

In [ ]:
dc_beta_genus <- betadisper(euc_dist_core_genus, sample_data(ps_core_genus_vst)$lpb_tertile)
dc_beta_genus$group.distances |> t()
anova(dc_beta_genus, by='margin')

In [ ]:
set.seed(08092003)
form_string <- paste("euc_dist_core_genus", "~", rhs)
f <- as.formula(form_string)
adonis2(f, meta, by = 'margin', permutations = how(blocks = meta$batch, nperm = 9999))

In [ ]:
(dc_pmnv <- adonis2(
    euc_dist_core_genus ~ is_lbp_normal + batch + d_sex,
    meta,
    by = "margin",
    permutations = how(blocks = meta$batch, nperm = 9999)))

# Alpha

In [ ]:
source('workflow/scripts/plots/alpha_violins.R')

## ASV level

In [ ]:
ps_core_no_zero_taxa <- prune_taxa(taxa_sums(ps_core) > 0, ps_core)
alpha <- estimate_richness(ps_core_no_zero_taxa, measures = c("Shannon", "Simpson", "Chao1"))
plot_alpha_violins(cbind(sample_data(ps_core), alpha), "lpb_tertile",  c("Shannon", "Simpson", "Chao1"))

## Genus level

In [ ]:
alpha <- estimate_richness(ps_core_genus, measures = c("Shannon", "Simpson", "Chao1"))
plot_alpha_violins(cbind(sample_data(ps_core_genus), alpha), 'lpb_tertile', c("Shannon", "Simpson", "Chao1"))

## Hill numbers

In [ ]:
source("workflow/scripts/plots/hill_curves.R")
options(repr.plot.width = 6, repr.plot.height = 5)
hill_res <- plot_hill_curves(
    ps_core,
    lpb_tertile,
    legend_title = "Depression vs. Control",
    q=seq(0, 5, by=0.25),
    summary=T,
    log_y=T
)


# Retest these results using CLR normalization and Bray-Curtis distances

In [ ]:
library(vegan)
library(phyloseq)
library(microbiome)

In [ ]:
set.seed(08092003)
ps_core_rare <- rarefy_even_depth(ps_core, rngseed = 08092003)

In [ ]:
bray_dist_core <- phyloseq::distance(ps_core_rare, method = "bray")

In [ ]:
options(repr.plot.width = 13, repr.plot.height = 13)
plot_dendrogram(
    bray_dist_core, ps_core_rare,
    lpb_tertile, d_sex, batch,
    taxa_rank = "genus", n_taxa = 15
)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 8)
plot_pcoa(ps_core_rare, bray_dist_core, lpb_tertile, legend_title="Depression vs. Control")

In [ ]:
dc_beta_bray <- betadisper(bray_dist_core, sample_data(ps_core_rare)$lpb_tertile)
dc_beta_bray$group.distances |> t()

anova(dc_beta_bray, by='margin')

set.seed(08092003)
form_string <- paste("bray_dist_core", "~", rhs)
f <- as.formula(form_string)
adonis2(f, meta, by = 'margin', permutations = 999)